# 02 · Launch TorchDistributor — Single Node

학습 로직은 [`torch_distributor_trainer.py`](./torch_distributor_trainer.py)의 `train_fn(...)`에 있습니다. 본 노트북은 driver 역할로 파라미터 구성·`TorchDistributor.run` 호출·결과 확인만 담당합니다.

| 섹션 | 모드 | 호출 |
|------|------|------|
| 1×1  | DDP, world_size=1 | `TorchDistributor(num_processes=1, local_mode=True).run(train_fn, ...)` |
| 1×N  | DDP, single node  | `TorchDistributor(num_processes=N, local_mode=True).run(train_fn, ...)` |

두 섹션이 같은 `train_fn`을 공유합니다. 다른 점은 `num_processes`뿐.

> `local_mode=True`는 child 프로세스를 driver 노드 안에서 띄운다는 뜻입니다. multi-node M×N은 [`03-launch_torch_distributor_multi_node.ipynb`](03-launch_torch_distributor_multi_node.ipynb).

**클러스터**: Single node, DBR 17.3 LTS ML, Driver `g5.12xlarge` (4× A10G), Workers 0, Autoscaling off. 1×1·1×N 모두 같은 클러스터에서 실행 가능.

In [ ]:
%run ./00-setup

## 학습 함수 import

`torch_distributor_trainer.py`를 driver의 `sys.path`에 올려두면 TorchDistributor가 cloudpickle로 `train_fn`을 child에 보낼 때 함수 본문이 그대로 직렬화됩니다. child 함수 내부에서도 `model.py`를 import하므로 `script_dir`을 함수 인자로 함께 전달합니다.

In [ ]:
import os
import sys

# Databricks Repos / Workspace files를 가리키는 절대 경로. driver와 worker 모두 동일 경로에 마운트.
NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
print(f"SCRIPT_DIR={SCRIPT_DIR}")

from torch_distributor_trainer import train_fn
print("train_fn imported:", train_fn.__module__)

## 1×1 GPU

`TorchDistributor(num_processes=1, local_mode=True).run(train_fn, ...)`. world_size=1에서 DDP all_reduce는 no-op이므로 결과는 1-GPU 학습과 같지만, launcher·학습 함수는 1×N·M×N과 일관됩니다.

In [ ]:
import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="td-1x1", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=1,
    local_mode=True,
    use_gpu=True,
).run(
    train_fn,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "td-1x1.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="1x1",
    script_dir=SCRIPT_DIR,
)

## 1×N GPU

같은 `train_fn`을 `num_processes=N`으로 호출. global batch는 `batch_size_per_gpu × N`으로 커집니다.

In [ ]:
from pyspark.ml.torch.distributor import TorchDistributor

NUM_GPUS_PER_NODE = 4  # 클러스터 driver의 GPU 수에 맞춰 조정

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="td-1xN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_GPUS_PER_NODE,
    local_mode=True,
    use_gpu=True,
).run(
    train_fn,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "td-1xN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="1xN",
    script_dir=SCRIPT_DIR,
)